In [19]:
import importlib
import sys
from pathlib import Path

import numpy as np
from numpy.testing import assert_array_equal

workspace_root = Path.cwd()
if not (workspace_root / "compression_knn").exists():
    workspace_root = workspace_root.parent
sys.path.insert(0, str(workspace_root))

import compression_knn.knn as knn_module

importlib.reload(knn_module)

from compression_knn.knn import CompressionKNNClassifier
from compression_knn.knn import CompressionKNNClassifierCV

print(f"workspace_root={workspace_root}")

workspace_root=d:\projects\random_projects\compression-knn


# Using the Public compression_knn API

This notebook is a compact smoke test for the package-level classifier API.

It covers:
- the README-style fruit example
- all supported compressors
- the built-in cross-validated estimator
- single-sample prediction behavior

In [20]:
X_train = np.array([
    "red, round, sweet",
    "orange, round, tangy",
    "red, oblong, sweet",
    "orange, oblong, tangy",
    "green, round, sour",
], dtype=str)
y_train = np.array(["Apple", "Orange", "Apple", "Orange", "Apple"], dtype=str)

X_test = np.array([
    "yellow, round, sweet",
    "green, round, sweet",
], dtype=str)
expected_predictions = np.array(["Apple", "Apple"], dtype=str)

print("train samples:")
for sample, label in zip(X_train, y_train):
    print(f"  {label:6s} | {sample}")
print("\nexpected_predictions=", expected_predictions.tolist())

train samples:
  Apple  | red, round, sweet
  Orange | orange, round, tangy
  Apple  | red, oblong, sweet
  Orange | orange, oblong, tangy
  Apple  | green, round, sour

expected_predictions= ['Apple', 'Apple']


## Example 1: Predict with Every Supported Compressor

In [21]:
for compressor_name in ["gzip", "bzip2", "lzma"]:
    classifier = CompressionKNNClassifier(
        n_neighbors=3,
        compressor=compressor_name,
        random_state=0,
    )
    classifier.fit(X_train, y_train)
    predictions = classifier.predict(X_test)
    assert_array_equal(predictions, expected_predictions)
    print(f"{compressor_name:10s} predictions={predictions.tolist()}")

gzip       predictions=['Apple', 'Apple']
bzip2      predictions=['Apple', 'Apple']
lzma       predictions=['Apple', 'Apple']


## Example 2: Select k with CompressionKNNClassifierCV

In [22]:
X_cv = np.array([
    "alpha apple crisp",
    "alpha apple tart",
    "beta orange tangy",
    "beta orange peel",
    "gamma banana mellow",
    "gamma banana split",
], dtype=str)

y_cv = np.array([
    "Apple",
    "Apple",
    "Orange",
    "Orange",
    "Banana",
    "Banana",
], dtype=str)

In [23]:
cv_classifier = CompressionKNNClassifierCV(
    n_neighbors=[1, 2],
    compressor="gzip",
    cv=2,
    random_state=0,
)
cv_classifier.fit(X_cv, y_cv)
cv_predictions = cv_classifier.predict(X_cv)
cv_train_accuracy = np.mean(cv_predictions == y_cv)

print(f"selected_n_neighbors={cv_classifier.n_neighbors_}")
print(f"best_score={cv_classifier.best_score_:.3f}")
print("cv_result_=")
print(np.round(cv_classifier.cv_result_, 3))
print(f"train_accuracy={cv_train_accuracy:.3f}")

selected_n_neighbors=1
best_score=1.000
cv_result_=
[[1.    1.   ]
 [0.333 0.333]]
train_accuracy=1.000


## Example 3: Single-Sample Prediction

In [24]:
single_sample = np.array(["yellow, round, sweet"], dtype=str)
single_prediction = classifier.predict(single_sample)

print("single_prediction=", single_prediction.tolist())
assert single_prediction.shape == (1,)

single_prediction= ['Apple']


The notebook should finish without assertion failures. If it does, the public text-classification API is working on the same toy patterns used in the repo tests and README.